# Labeling the Dataset using the trained student

What we do:
1) Run the student model on the dataset of inputs
2) Analyse the student dataset and label it


Student: Qwen2.5-1.5B-Instruct

In [1]:
import dotenv

dotenv.load_dotenv()

True

In [2]:
from core.types import *
from core.utils.huggingface_client import HuggingFaceClient
from core.utils.huggingface_inference_client import HuggingFaceInferenceClient
from core.utils.ollama_inference_client import OllamaInferenceClient
from core.utils.openai_client import OpenAIClient, ProcessingMode
from doom.preprocessing.doom_game_state_perturbator import DoomGameStatePerturbator
from doom.utils.doom_game_state import DoomGameState, MonsterType, WeaponName, AimedAtType
from sklearn.cluster import DBSCAN
from dataclasses import dataclass, asdict
from collections import Counter
from typing import Iterable
from pathlib import Path
from ollama import ChatResponse
from openai.types.responses import Response as OpenAIResponse
from transformers import AutoTokenizer

import os
import json
import numpy as np
import pandas as pd

In [3]:
%load_ext autoreload
%autoreload 2

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("./models/training2/final")
tokenizer = AutoTokenizer.from_pretrained("./models/training2/final")

# Test inference
prompt = """
You are a game command parser that converts natural language commands into DSL instructions.
Game State:
AIMED_AT:
  type: Wall
  distance: 330.86
  interactable: yes

MONSTERS (count=0):

INVENTORY:
  current_slot: 2
  weapons:
    - (1, Fist, 0)
    - (2, Pistol, 50)
Command:
Go press that switch ahead
"""
inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)
# Don't decode the whole output - skip the input tokens
generated_tokens = outputs[0][len(inputs.input_ids[0]):]
print(tokenizer.decode(generated_tokens))

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Command: 330.76
  interactable: no

INVENTORY:
  current_slot: 2
  weapons:
    - (1, Fist, 0)
    - (2, Pistol


In [4]:
inputs = LLMCommandingInput.load_inputs(
    path=Path("data/inputs/inputs.json"),
    gstype=DoomGameState
)

inputs_lookup = {inp.id: inp for inp in inputs}

In [5]:
# For now, only extract inputs to label
selected_inputs = [
    inp
    for inp in inputs
    if inp.selected_for_labelling
]

print(f"Selected inputs: {len(selected_inputs)}/{len(inputs)}")

Selected inputs: 50/2872


In [ ]:
student_client = HuggingFaceInferenceClient[LLMCommandingInput, LLMCommandingOutput](
    model="./models/training2/final",
    max_output_tokens=800,
    temperature=0.0,
)

In [20]:
# Prepare Prompt (same as training)
system_prompt = "You are a game command parser that converts natural language commands into DSL instructions."

In [21]:
def format_input(inp: LLMCommandingInput) -> str:
    game_state = inp.game_state.state.to_prompt_ready()
    command = inp.user_command.command.command

    return f"Game State:\n{game_state}\nCommand:\n{command}"


def parse_output(response: ChatResponse, input_id: str, latency: float) -> LLMCommandingOutput:
    return LLMCommandingOutput(
        input_id=input_id,
        actions=response.message.content,
        reason=None,
        latency=latency,
    )


def get_id(gse: LLMCommandingInput, idx: int) -> str:
    return gse.id

In [22]:
print(system_prompt)
print(format_input(inputs[3]))

You are a game command parser that converts natural language commands into DSL instructions.
Game State:
AIMED_AT:
  type: Wall
  distance: 330.86
  interactable: yes

MONSTERS (count=0):

INVENTORY:
  current_slot: 2
  weapons:
    - (1, Fist, 0)
    - (2, Pistol, 50)
Command:
Go press that switch ahead


In [23]:
outputs = student_client.process(
    dataset=selected_inputs,
    system_prompt=system_prompt,
    tools = [], # No tools at level 3
    format_input=format_input,
    parse_output=parse_output,
    get_id=get_id,
)

🔄 Processing 50 items sequentially


Processing items: 100%|██████████| 50/50 [00:08<00:00,  5.59it/s]


✅ Completed: 50/50 successful


In [24]:
# Clustering was already made earlier, so inputs are already partitioned.
# Now, considering this is just a test of the Teacher's quality (to save time):
# - Having retrieved the LLMCommandingOutputs, I can just prepare the csv
# - This time, every row in the csv should be set to be evaluated.
# For the full run: check if selected. (MAKE SURE TO CHANGE FILE NAME SO THAT I DO NOT HAVE TO REVALUATE IF THEY ALREADY CORRECT)

rows = []
for idx, output in enumerate(outputs):
    inp = inputs_lookup[output.input_id]

    row = LLMCommandingLabelledDataPoint(
        input_id=inp.id,
        game_state=inp.game_state.state.to_prompt_ready(),
        command=inp.user_command.command.command,
        command_intent=inp.user_command.command.intent,
        command_explicitness=inp.user_command.command.explicitness,
        command_atomicity=float(inp.user_command.command.atomicity),
        command_contextuality=float(inp.user_command.command.contextuality),
        game_actions=output.actions.__str__(),
        latency=output.latency,
        reason_if_failed=output.reason,
        cluster_id=inp.cluster_id,
        selected_for_labelling=inp.selected_for_labelling,
    )

    rows.append(asdict(row))

df = pd.DataFrame(rows)

In [25]:
output_path = Path("data/outputs/selected-data-training-fgemma-q4.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)

In [6]:
tokenizer = AutoTokenizer.from_pretrained("./models/training2/final")

# Check how it tokenizes your commands
test = "ROTATE_TO_TARGET MONSTER_1"
tokens = tokenizer.tokenize(test)
print(tokens)
print(tokenizer.convert_tokens_to_ids(tokens))

['ROTATE', '_', 'TO', '_', 'TARGET', '▁MON', 'STER', '_', '1']
[215609, 236779, 6257, 236779, 54637, 31540, 32867, 236779, 236770]


In [7]:
tokenizer = AutoTokenizer.from_pretrained("google/functiongemma-270m-it")

# Check how it tokenizes your commands
test = "ROTATE_TO_TARGET MONSTER_1"
tokens = tokenizer.tokenize(test)
print(tokens)
print(tokenizer.convert_tokens_to_ids(tokens))

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

C:\Users\Filippo Corti\Documents\GitHub\GamePals-LLM-Distillation\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Filippo Corti\.cache\huggingface\hub\models--google--functiongemma-270m-it. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' packa

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/63.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/13.8k [00:00<?, ?B/s]

['ROTATE', '_', 'TO', '_', 'TARGET', '▁MON', 'STER', '_', '1']
[215609, 236779, 6257, 236779, 54637, 31540, 32867, 236779, 236770]


In [8]:
tokenizer = AutoTokenizer.from_pretrained("./models/training2/final")

# Test exact tokenization
text = "ROTATE_TO_TARGET MONSTER_1"
tokens = tokenizer.encode(text, add_special_tokens=False)
decoded = tokenizer.decode(tokens)

print(f"Original: '{text}'")
print(f"Decoded:  '{decoded}'")
print(f"Tokens: {tokens}")
print(f"Token strings: {[tokenizer.decode([t]) for t in tokens]}")

Original: 'ROTATE_TO_TARGET MONSTER_1'
Decoded:  'ROTATE_TO_TARGET MONSTER_1'
Tokens: [215609, 236779, 6257, 236779, 54637, 31540, 32867, 236779, 236770]
Token strings: ['ROTATE', '_', 'TO', '_', 'TARGET', ' MON', 'STER', '_', '1']
